kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import anndata as ad
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "gao"
MOUSE_SERIES = SERIES + "_mouse"
HUMAN_SERIES = SERIES + "_human"

clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")
raw_data_dir = find_env_dir("RAW_DATA")

In [2]:
matrix_file = os.path.join(raw_data_dir, "su", "exprMatrix.tsv.gz")
meta_file = os.path.join(raw_data_dir, "su", "meta.tsv")

obs = pd.read_csv(meta_file, sep="\t").set_index("Cell_ID")

gene_names = []
chunks = []

for x in pd.read_csv(matrix_file, sep="\t", index_col=0, chunksize=128):
    gene_names.extend(x.index.astype(str))
    chunks.append(csr_matrix(x.values.T))

X = hstack(chunks, format="csr")

cell_ids = pd.read_csv(matrix_file, sep="\t", nrows=0, index_col=0).columns
obs = obs.loc[cell_ids]

gao_human = ad.AnnData(X=X, obs=obs)
gao_human.var_names = gene_names
gao_human.var_names_make_unique()

In [5]:
assert isinstance(gao_human.obs, pd.DataFrame)
gao_human.obs["sample"] = gao_human.obs["Sample"]
gao_human.obs["donor"] = gao_human.obs["sample"]
gao_human.obs["sex"] = gao_human.obs["Sex"]
gao_human.obs["age"] = gao_human.obs["Age"]
gao_human.obs["batch"] = gao_human.obs["Batch"]
gao_human.obs["agegroup"] = gao_human.obs["AgeGroup_5"]
gao_human.obs["celltype"] = gao_human.obs["MajorCellTypes"]
gao_human.obs["series"] = "su"

keep = ["sample", "donor", "sex", "age", "batch", "agegroup", "celltype", "series"]
gao_human.obs.drop(columns = [c for c in gao_human.obs.columns if c not in keep], inplace=True) #type: ignore

In [6]:
gao_human.write(os.path.join(pre_h5ad_dir, HUMAN_SERIES + "_raw.h5ad"))

In [2]:
mouse = ad.read_h5ad(os.path.join(clustered_h5ad_dir, MOUSE_SERIES + ".h5ad"))
human = ad.read_h5ad(os.path.join(clustered_h5ad_dir, HUMAN_SERIES + ".h5ad"))

In [78]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "FERMT1", "CACNG5", "VCAN",],
    "COP": ["GPR17", "ENPP6", "BCAS1", ],
    "Oligodendrocyte": ["MAG", "CNP", "MOBP", "ERMN", "GPR37", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", "GRIN1", "GRIN2B", "MYT1L", ],
    "Excitatory": ["SLC17A7", "NEUROD2", "SATB2", "TBR1", "RORB", "SLC17A6", "DRD1", ],
    "Inhibitory": ["GAD1", "GAD2", "SLC6A1", "SLC32A1", "DLX1", "DLX2", "DLX5", "DLX6", ],
    "CRcell": ["RELN", "TP73", "CXCR4", ], #Cajal-Retzius Cell
    "Microglia": ["P2RY12", "TMEM119", "SLC2A5", "SELPLG", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", ],
    "Endothelial": ["CLDN5", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "AGT", "ID4", "RFX4", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "TPPP3", "RSPH1", "PIFO", ],
    "CPEC": ["KCNJ13", "SLC13A4", "SLC39A12", "CLIC6", "TTR", "SLC5A5", "FOLR1", "AQP1", ], #Choroid Plexus Epithelial Cell
    "VSMC": ["TAGLN", "ACTA2", "MYH11", ], #Vascular
    "VLMC": ["COL1A2", "DCN", ],
    "RBC": ["HBB", "HBA1", "HBA2", ],
    "LowQuality": ["MALAT1", "NEAT1", ]
}

plot.plot_dotplot(human, HUMAN_SERIES, cell_marker_genes, group = "leiden")

In [ ]:
cell_marker_genes = {
    "OPC": ["Pdgfra", "Megf11", "Vcan", "Myt1", "Cdo1", ],
    "COP": ["Gpr17", "Has2", "9630013A20Rik", "Col9a1", "Bmp4", "Tcf7l2", "Bcas1",],
    "Oligodendrocyte": ["Mbp", "Mog", "Mag", "Cnp", "Mobp", "Fa2h", "Ppp1r14a", "Ermn", "Gpr37", "Myrf", ],
    "IPC": ["Gadd45g", "Hmgb2", "Ascl1", "Insm1", "Tead2", "Dll1", "Dll3", ],
    "Neuroblast": ["Sox11", "Sox4", "Tubb3", "Dcx", "Ccnd2", "Draxin", "Igfbpl1", ],
    "Neuron": ["Snap25", "Meg3", "Rbfox3", "Eno2", "Syt1", "Grin1", "Tmem130", ],
    "Excitatory": ["Slc17a7", "Neurod2", "Satb2", "Tbr1", ],
    "Inhibitory": ["Gad1", "Gad2", "Dlx1", "Dlx2", "Dlx5", "Dlx6", "Slc32a1", ],
    "Microglia": ["Tmem119", "Siglech", "Olfml3", "Gpr34", "Ctss", "Cd53" ],
    "Endothelial": ["Cldn5", "Cd34", "Flt1", "Vwf", "Cdh5", "Clec14a", "Erg", "Itm2a", ], #Vascular
    "Pericyte": ["Rgs5", "Cd248", "Kcnj8", ], #Vascular
    "Astrocyte": ["Aqp4", "Aldh1l1", "Atp1b2", "Aldh1a1", "Atp13a4", "Bmpr1b", "Rfx4", "Clu", ],
    "Ependymal": ["Drd1", "Tac1", "Pdyn", "Isl1", "Ebf1", "Ppp1r1b", "Drd2", "Adora2a", "Penk", "Gpr6", "Sp9", ],
    "VLMC": ["Dcn", "Lum", "Col1a1", "Col1a2", "Col3a1", ],
}

plot.plot_dotplot(mouse, MOUSE_SERIES, cell_marker_genes, group = "leiden")

In [3]:
leiden_to_celltype = {
    "0": "ExcitatoryNeuron",
    "1": "InhibitoryNeuron",
    "2": "ExcitatoryNeuron",
    "3": "Astrocyte",
    "4": "ExcitatoryNeuron",
    "5": "Vascular+VLMC",
    "6": "ExcitatoryNeuron",
    "7": "ExcitatoryNeuron",
    "8": "ExcitatoryNeuron",
    "9": "ExcitatoryNeuron",
    "10": "Doublet",
    "11": "ExcitatoryNeuron",
    "12": "ExcitatoryNeuron",
    "13": "ExcitatoryNeuron",
    "14": "ExcitatoryNeuron",
    "15": "InhibitoryNeuron",
    "16": "InhibitoryNeuron",
    "17": "Astrocyte",
    "18": "ExcitatoryNeuron",
    "19": "ExcitatoryNeuron",
    "20": "ExcitatoryNeuron",
    "21": "Oligodendrocyte",
    "22": "ExcitatoryNeuron",
    "23": "Doublet",
    "24": "ExcitatoryNeuron",
    "25": "ExcitatoryNeuron",
    "26": "ExcitatoryNeuron",
    "27": "ExcitatoryNeuron",
    "28": "ExcitatoryNeuron",
    "29": "ExcitatoryNeuron",
    "30": "ExcitatoryNeuron",
    "31": "InhibitoryNeuron",
    "32": "InhibitoryNeuron",
    "33": "ExcitatoryNeuron",
    "34": "ExcitatoryNeuron",
    "35": "Oligodendrocyte",
    "36": "OPC",
    "37": "ExcitatoryNeuron",
    "38": "ExcitatoryNeuron",
    "39": "InhibitoryNeuron",
    "40": "ExcitatoryNeuron",
    "41": "Astrocyte",
    "42": "ExcitatoryNeuron",
    "43": "COP",
    "44": "ExcitatoryNeuron",
    "45": "Astrocyte",
    "46": "OPC",
    "47": "Doublet",
    "48": "Ependymal",
    "49": "ExcitatoryNeuron",
    "50": "InhibitoryNeuron",
    "51": "ExcitatoryNeuron",
    "52": "Astrocyte",
    "53": "ExcitatoryNeuron",
    "54": "ExcitatoryNeuron",
    "55": "ExcitatoryNeuron",
    "56": "Astrocyte",
    "57": "ExcitatoryNeuron",
    "58": "CPEC",
    "59": "ExcitatoryNeuron",
    "60": "InhibitoryNeuron",
    "61": "ExcitatoryNeuron",
    "62": "ExcitatoryNeuron",
    "63": "ExcitatoryNeuron", 
    "64": "ExcitatoryNeuron",
    "65": "InhibitoryNeuron",
    "66": "ExcitatoryNeuron",
    "67": "CRcell",
    "68": "ExcitatoryNeuron",
    "69": "ExcitatoryNeuron",
    "70": "ExcitatoryNeuron",
}
leiden_str = human.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
human.obs["cell"] = mapped
human = human[human.obs["cell"] != "LowQuality"].copy()

In [4]:
leiden_to_celltype = {
    "0": "ExcitatoryNeuron",
    "1": "ExcitatoryNeuron",
    "2": "ExcitatoryNeuron",
    "3": "ExcitatoryNeuron",
    "4": "ExcitatoryNeuron",
    "5": "ExcitatoryNeuron",
    "6": "InhibitoryNeuron",
    "7": "InhibitoryNeuron",
    "8": "ExcitatoryNeuron",
    "9": "InhibitoryNeuron",
    "10": "InhibitoryNeuron",
    "11": "ExcitatoryNeuron",
    "12": "ExcitatoryNeuron",
    "13": "ExcitatoryNeuron",
    "14": "ExcitatoryNeuron",
    "15": "ExcitatoryNeuron",
    "16": "InhibitoryNeuron",
    "17": "ExcitatoryNeuron",
    "18": "ExcitatoryNeuron",
    "19": "ExcitatoryNeuron",
    "20": "ExcitatoryNeuron",
    "21": "VLMC",
    "22": "ExcitatoryNeuron",
    "23": "ExcitatoryNeuron",
    "24": "InhibitoryNeuron",
    "25": "InhibitoryNeuron",
    "26": "ExcitatoryNeuron",
    "27": "Oligodendrocyte",
    "28": "Astrocyte",
    "29": "Endothelial",
    "30": "InhibitoryNeuron",
    "31": "Oligodendrocyte",
    "32": "InhibitoryNeuron",
    "33": "InhibitoryNeuron",
    "34": "InhibitoryNeuron",
    "35": "LowQuality",
    "36": "ExcitatoryNeuron",
    "37": "Oligodendrocyte",
    "38": "InhibitoryNeuron",
    "39": "InhibitoryNeuron",
    "40": "InhibitoryNeuron",
    "41": "InhibitoryNeuron",
    "42": "Neuroblast",
    "43": "Oligodendrocyte",
    "44": "InhibitoryNeuron",
    "45": "Astrocyte",
    "46": "InhibitoryNeuron",
    "47": "ExcitatoryNeuron",
    "48": "Oligodendrocyte",
    "49": "ExcitatoryNeuron",
    "50": "InhibitoryNeuron",
    "51": "Oligodendrocyte",
    "52": "Oligodendrocyte",
    "53": "Astrocyte",
    "54": "Astrocyte",
    "55": "InhibitoryNeuron",
    "56": "IPC",
    "57": "InhibitoryNeuron",
    "58": "?",
    "59": "Oligodendrocyte",
    "60": "Astrocyte",
    "61": "ExcitatoryNeuron",
    "62": "Oligodendrocyte",
    "63": "ExcitatoryNeuron", 
    "64": "InhibitoryNeuron",
    "65": "OPC",
    "66": "InhibitoryNeuron",
    "67": "Pericyte",
    "68": "ExcitatoryNeuron",
    "69": "ExcitatoryNeuron",
    "70": "COP",
    "71": "Microglia",
    "72": "InhibitoryNeuron",
    "73": "Oligodendrocyte",
    "74": "Microglia",
    "75": "ExcitatoryNeuron",
}
leiden_str = mouse.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
mouse.obs["cell"] = mapped
mouse = mouse[mouse.obs["cell"] != "LowQuality"].copy()

In [81]:
human.obs["celltype"] = human.obs["cell"].copy() #type: ignore
plot.plot_umap(human, HUMAN_SERIES, has_celltype=True)

In [ ]:
mouse.obs["celltype"] = mouse.obs["cell"].copy() #type: ignore
plot.plot_umap(mouse, MOUSE_SERIES, has_celltype=True)

In [5]:
assert isinstance(mouse.obs, pd.DataFrame)
mouse.obs["series"] = mouse.obs["project_id"]
mouse.obs["sample"] = mouse.obs["sample"]
mouse.obs["donor"] = mouse.obs["sample"]
mouse.obs["celltype"] = mouse.obs["cell"]
mouse.obs["cell"] = mouse.obs["cell"]
mouse.obs["leiden"] = mouse.obs["leiden"]

keep = ["series", "sample", "donor", "celltype", "cell", "leiden"]
mouse.obs.drop(columns = [c for c in mouse.obs.columns if c not in keep], inplace=True) #type: ignore
mouse.write_h5ad(os.path.join(final_h5ad_dir, MOUSE_SERIES + "_final.h5ad"))

In [9]:
human.obs

,sample,donor,sex,age,batch,agegroup,celltype,series,n_genes_by_counts,total_counts,...,total_counts_mt,pct_counts_mt,log1p_total_counts_mt,total_counts_ribo,pct_counts_ribo,log1p_total_counts_ribo,n_counts,n_genes,leiden,cell
AAACATCGAACAACCACACTTCGA_1_1_1_1_1_1,S43,S43,M,0,B1,Infancy,Glut.N,su,2295,4659.0,...,0.0,0.0,0.0,19.0,0.407813,2.995732,4659.0,2295,37,ExcitatoryNeuron
AAACATCGAACAACCACAGATCTG_1_1_1_1_1_1,S43,S43,M,0,B1,Infancy,AST,su,767,1001.0,...,0.0,0.0,0.0,4.0,0.399600,1.609438,1001.0,767,3,Astrocyte
AAACATCGAACGCTTAAAACATCG_1_1_1_1_1_1,S43,S43,M,0,B1,Infancy,Glut.N,su,1882,3439.0,...,0.0,0.0,0.0,10.0,0.290782,2.397895,3439.0,1882,68,ExcitatoryNeuron
AAACATCGAACGCTTAAACGTGAT_1_1_1_1_1_1,S43,S43,M,0,B1,Infancy,GABA.N,su,1088,1866.0,...,0.0,0.0,0.0,5.0,0.267953,1.791759,1866.0,1088,16,InhibitoryNeuron
AAACATCGAACGTGATCCAGTTCA_1_1_1_1_1_1,S43,S43,M,0,B1,Infancy,Glut.N,su,2256,4555.0,...,0.0,0.0,0.0,24.0,0.526893,3.218876,4555.0,2256,37,ExcitatoryNeuron
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTCACGCATCTTCACAACGCTCGA_1_1_2,S52,S52,M,85,B12,Aging,Glut.N,su,1504,2251.0,...,0.0,0.0,0.0,2.0,0.088849,1.098612,2251.0,1504,2,ExcitatoryNeuron
TTCACGCATGAAGAGAAACAACCA_1_1_2,S52,S52,M,85,B12,Aging,OPC,su,1914,3171.0,...,0.0,0.0,0.0,11.0,0.346894,2.484907,3171.0,1914,36,OPC
TTCACGCATGGCTTCAAAGGTACA_1_1_2,S52,S52,M,85,B12,Aging,Glut.N,su,1589,2511.0,...,0.0,0.0,0.0,12.0,0.477897,2.564949,2511.0,1589,22,ExcitatoryNeuron
TTCACGCATGGCTTCAACGTATCA_1_1_2,S52,S52,M,85,B12,Aging,AST,su,907,1259.0,...,0.0,0.0,0.0,3.0,0.238284,1.386294,1259.0,907,45,Astrocyte


In [10]:
assert isinstance(human.obs, pd.DataFrame)
human.obs["sample"] = human.obs["sample"]
human.obs["donor"] = human.obs["donor"]
human.obs["sex"] = human.obs["sex"]
human.obs["age"] = human.obs["age"]
human.obs["batch"] = human.obs["batch"]
human.obs["agegroup"] = human.obs["agegroup"]
human.obs["celltype"] = human.obs["cell"]
human.obs["cell"] = human.obs["cell"]
human.obs["leiden"] = human.obs["leiden"]
human.obs["series"] = human.obs["series"]

keep = ["sample", "donor", "sex", "age", "batch", "agegroup", "celltype", "cell", "leiden", "series"]
human.obs.drop(columns = [c for c in human.obs.columns if c not in keep], inplace=True) #type: ignore
human.write_h5ad(os.path.join(final_h5ad_dir, HUMAN_SERIES + "_final.h5ad"))

In [82]:
oligo_human = human[
    (human.obs["cell"] == "OPC") |
    (human.obs["cell"] == "COP") | 
    (human.obs["cell"] == "Oligodendrocyte")
]
oligo_human.write_h5ad(os.path.join(pre_h5ad_dir, HUMAN_SERIES + "_oligo_raw.h5ad"))

/tmp/ipykernel_1996804/1408120544.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  oligo_human.write_h5ad(os.path.join(pre_h5ad_dir, HUMAN_SERIES + "_oligo_raw.h5ad"))
/tmp/ipykernel_1996804/1408120544.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  oligo_human.write_h5ad(os.path.join(pre_h5ad_dir, HUMAN_SERIES + "_oligo_raw.h5ad"))


In [ ]:
oligo_mouse = mouse[
    (mouse.obs["cell"] == "OPC") |
    (mouse.obs["cell"] == "COP") | 
    (mouse.obs["cell"] == "Oligodendrocyte")
]
oligo_mouse.write_h5ad(os.path.join(pre_h5ad_dir, MOUSE_SERIES + "_oligo_raw.h5ad"))